### ETL - PIPELINE (WITH LOGGING)

In [4]:
import pandas as pd
from sqlalchemy import create_engine
from datetime import datetime

from utils.config import *
from utils.db import *

import logging # thu vien mac dinh cua Python (luu log+xem gio chay)
import os # thu vien operation system (dung de vao may de xem xet gio giac, file)

In [5]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
    handlers=[
        logging.FileHandler(
            f"log/ETL_pipeline_advance_{datetime.now():%Y%m%d}.log",
            encoding='utf-8'
        ), ## File log,
        logging.StreamHandler() ## In ra terminal cho xem
    ]
)

log = logging.getLogger(__name__)


def extract():
    log.info("EXTRACT started!")

    df_a = pd.read_csv('rawdata/ETL Pipeline data/ad_metrics_2.csv')
    log.info(f"Get data ad done: {len(df_a)} rows")

    df_e = pd.read_csv('rawdata/ETL Pipeline data/employees_2.csv')
    log.info(f"Get data employees done: {len(df_e)} rows")

    log.info("EXTRACT done!")
    
    return df_a,df_e



def transform(df_a,df_e):
    log.info("TRANSFORM started!")

    # df_a setting
    df_a = df_a.copy() # df_e va df_a hien tai chi la mot ket qua tham chieu chu khong phai table (Setting CopyWarning)
    ## Chuyen datetime
    df_a["date"] = pd.to_datetime(df_a["date"])
    ## them cot
    df_a["cost_by_click"] = df_a["ad_spend"] / df_a["clicks"]
    df_a["dtime_data_created"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log.info("TRANSFORM data ads done!")


    # df_e setting
    df_e = df_e.copy() # df_e va df_a hien tai chi la mot ket qua tham chieu chu khong phai table (Setting CopyWarning)
    ## Chuyen datetime
    df_e["enroll_date"] = pd.to_datetime(df_e["enroll_date"])
    ## fill null
    df_e["full_name"] = df_e["full_name"].fillna("Unknown")
    df_e["fee_paid"] = df_e["fee_paid"].fillna(df_e["fee_paid"].median())
    ## them cot
    df_e["enroll_month"] = df_e["enroll_date"].dt.month
    df_e["dtime_data_created"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log.info("TRANSFORM data employees done!")


    log.info("TRANSFORM end!")

    return df_a,df_e



def load(df_a,df_e):
    log.info("LOAD started!")

    engine = make_engine(INTRO_DE)

    df_a.to_sql("fact_ad",engine ,if_exists = "append" ,index= False)
    df_e.to_sql("fact_employee",engine ,if_exists = "append" ,index= False)

    log.info("LOAD end!")

In [6]:
df_a,df_e = extract()
df_a,df_e = transform(df_a,df_e)
load(df_a,df_e)

10:29:33 | INFO | EXTRACT started!
10:29:33 | INFO | Get data ad done: 182 rows
10:29:33 | INFO | Get data employees done: 300 rows
10:29:33 | INFO | EXTRACT done!
10:29:33 | INFO | TRANSFORM started!
10:29:33 | INFO | TRANSFORM data ads done!
10:29:33 | INFO | TRANSFORM data employees done!
10:29:33 | INFO | TRANSFORM end!
10:29:33 | INFO | LOAD started!
10:29:33 | INFO | LOAD end!
